In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_identification_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyIdentificationId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "PartyIdentificationKey string,"
        #     "PartyKey string,"
        #     "CDI string,"
        #     "PartyIdentificationTypeCode string,"
        #     "PartyIdentificationType string,"
        #     "PartyIdentificationNumber string,"
        #     "SourceTable string,"
        #     "MainFlag string,"
        #     "IssueAuthority string,"
        #     "IssueDate date,"
        #     "ExpiredDate date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.PartyIdentification"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qc",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "CUST_CODE",
            "doc_id_num",
            "doc_id_type",
            "vendor_code",
            "CDI",
            "businessdate",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",
            customer_final_filters,
            customer_selected_fields,
        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "vendor_code_qualco",
            "portfolioname",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== Join ==================
        PartyIdentificationId_primary_key_fields = [
            customer.CUST_CODE,
            customer.doc_id_num,
            customer.doc_id_type,
            lit("qc_customer"),
            lit("qc"),
        ]

        PartyIdentificationKey_primary_key_fields = [
            customer.CUST_CODE,
            customer.doc_id_num,
            customer.doc_id_type,
            lit("qc_customer"),
        ]
        party_identification_qc = customer.join(
            portfolio,
            customer.vendor_code == portfolio.vendor_code_qualco,
            "left",
        ).where(customer.doc_id_num.isNotNull() & customer.CDI.isNotNull())
        # ================== party_identification_qc ==================
        party_identification_qc = party_identification_qc.withColumn(
            "PartyIdentificationId",
            sha2(concat_ws(";", *PartyIdentificationId_primary_key_fields), 256),
        ).withColumn(
            "PartyIdentificationKey",
            concat_ws(";", *PartyIdentificationKey_primary_key_fields),
        )

        party_identification_qc = party_identification_qc.select(
            party_identification_qc.PartyIdentificationId.cast("string").alias(
                "PartyIdentificationId"
            ),
            lit("qc").alias("SourceSystem"),
            party_identification_qc.vendor_code.cast("string").alias("PortfolioKey"),
            party_identification_qc.portfolioname.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            party_identification_qc.PartyIdentificationKey.cast("string").alias(
                "PartyIdentificationKey"
            ),
            party_identification_qc.CUST_CODE.cast("string").alias("PartyKey"),
            party_identification_qc.CDI.cast("string").alias("CDI"),
            party_identification_qc.doc_id_type.cast("string").alias(
                "PartyIdentificationTypeCode"
            ),
            party_identification_qc.doc_id_type.cast("string").alias(
                "PartyIdentificationType"
            ),
            party_identification_qc.doc_id_num.cast("string").alias(
                "PartyIdentificationNumber"
            ),
            lit("qc_customer").cast("string").alias("SourceTable"),
            lit(None).cast("string").alias("MainFlag"),
            lit(None).cast("string").alias("IssueAuthority"),
            lit(None).cast("date").alias("IssueDate"),
            lit(None).alias("ExpiredDate"),
            party_identification_qc.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        party_identification_qc.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_party_identification_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)